In [ ]:
import os
import json
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
import json

train_json = []

with open(r"C:\Users\Fixity\OneDrive - Fixity Technologies\Desktop\2nd phase projects\Second 50 Projects Vikas\rest projects\A CNN-Based Approach for Classifying Traditional and Modern Indo-Fashion Styles\images\train_data.json","r") as f:
    
    for line in f:
        train_json.append(json.loads(line))

print("Total train records:",len(train_json))

Total train records: 91166


In [ ]:
val_json = []
with open(r"C:\Users\Fixity\OneDrive - Fixity Technologies\Desktop\2nd phase projects\Second 50 Projects Vikas\rest projects\A CNN-Based Approach for Classifying Traditional and Modern Indo-Fashion Styles\images\val_data.json","r") as f:
    for line in f:
        val_json.append(json.loads(line))


test_json = []
with open(r"C:\Users\Fixity\OneDrive - Fixity Technologies\Desktop\2nd phase projects\Second 50 Projects Vikas\rest projects\A CNN-Based Approach for Classifying Traditional and Modern Indo-Fashion Styles\images\test_data.json","r") as f:
    for line in f:
        test_json.append(json.loads(line))

In [ ]:
all_data = train_json + val_json + test_json
print("Total records:", len(all_data))

Total records: 106166


In [ ]:
labels = set()

for item in all_data:
    labels.add(item["class_label"])

labels = sorted(list(labels))

print("Total labels:",len(labels))
print(labels)

Total labels: 15
['blouse', 'dhoti_pants', 'dupattas', 'gowns', 'kurta_men', 'leggings_and_salwars', 'lehenga', 'mojaris_men', 'mojaris_women', 'nehru_jackets', 'palazzos', 'petticoats', 'saree', 'sherwanis', 'women_kurta']


In [ ]:
label_to_index = {label:i for i,label in enumerate(labels)}
print("Label to Index Mapping:", label_to_index)

Label to Index Mapping: {'blouse': 0, 'dhoti_pants': 1, 'dupattas': 2, 'gowns': 3, 'kurta_men': 4, 'leggings_and_salwars': 5, 'lehenga': 6, 'mojaris_men': 7, 'mojaris_women': 8, 'nehru_jackets': 9, 'palazzos': 10, 'petticoats': 11, 'saree': 12, 'sherwanis': 13, 'women_kurta': 14}


In [ ]:
print(type(train_json))
print(len(train_json))
print(train_json[0])

<class 'list'>
91166
{'image_url': 'https://m.media-amazon.com/images/I/81XKaSKvlyL._AC_UL320_.jpg', 'image_path': 'images/train/0.jpeg', 'brand': 'Womanista', 'product_title': "Women's Georgette Saree with Blouse Piece (TKIM811_Black_Free Size)", 'class_label': 'saree'}


In [ ]:
labels = sorted(list(set(item["class_label"] for item in all_data)))

print("Total classes:", len(labels))
print(labels)

Total classes: 15
['blouse', 'dhoti_pants', 'dupattas', 'gowns', 'kurta_men', 'leggings_and_salwars', 'lehenga', 'mojaris_men', 'mojaris_women', 'nehru_jackets', 'palazzos', 'petticoats', 'saree', 'sherwanis', 'women_kurta']


In [ ]:
import os

image_paths = []
image_labels = []

for item in all_data:

    path = item["image_path"]
    label = item["class_label"]

    if os.path.exists(path):
        image_paths.append(path)
        image_labels.append(label_to_index[label])

In [ ]:
print(len(image_paths))

106166


In [ ]:
from sklearn.model_selection import train_test_split

X_train, _, y_train, _ = train_test_split(
    image_paths,
    image_labels,
    train_size=0.65,
    stratify=image_labels,
    random_state=42
)

print("Training samples:", len(X_train))

Training samples: 69007


In [ ]:
import tensorflow as tf

def preprocess(path,label):

    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img,channels=3)
    img = tf.image.resize(img,(96,96))
    img = img/255.0

    return img,label

dataset = tf.data.Dataset.from_tensor_slices((X_train,y_train))
dataset = dataset.map(preprocess).batch(64).prefetch(tf.data.AUTOTUNE)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, _, y_train, _ = train_test_split(
    image_paths,
    image_labels,
    train_size=0.65,
    stratify=image_labels,
    random_state=42
)

print("Training samples:", len(X_train))

Training samples: 69007


In [ ]:
import tensorflow as tf

dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))

In [ ]:
def preprocess(path, label):

    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (96,96))
    img = img / 255.0

    return img, label

In [ ]:
dataset = dataset.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

dataset = dataset.shuffle(10000)

dataset = dataset.batch(64)

dataset = dataset.prefetch(tf.data.AUTOTUNE)

In [ ]:
for img, label in dataset.take(1):
    print(img.shape)

(64, 96, 96, 3)


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
num_classes = len(labels)

print("Total classes:", num_classes)

Total classes: 15


In [ ]:
import tensorflow as tf
import pandas as pd
import json
import os
from sklearn.model_selection import train_test_split

# ==============================
# PARAMETERS
# ==============================

IMG_SIZE = (96,96)
BATCH_SIZE = 64
EPOCHS = 50

# ==============================
# LOAD JSON LINES DATA
# ==============================

data = []

with open(r"images/train_data.json","r") as f:
    for line in f:
        data.append(json.loads(line))

df = pd.DataFrame(data)

df = df[["image_path","class_label"]]

print("Total Images Before Sampling:",len(df))

# ==============================
# USE 50% DATA FROM EACH CLASS
# ==============================

df = df.groupby("class_label").apply(lambda x: x.sample(frac=0.5,random_state=42)).reset_index(drop=True)

print("Total Images After 50% Sampling:",len(df))

# ==============================
# VERIFY IMAGE PATH
# ==============================

print("Sample Path:",df["image_path"].iloc[0])
print("Exists:",os.path.exists(df["image_path"].iloc[0]))

# ==============================
# LABEL ENCODING
# ==============================

class_names = df["class_label"].unique()

label_map = {name:i for i,name in enumerate(class_names)}

df["label"] = df["class_label"].map(label_map)

num_classes = len(class_names)

print("Classes:",class_names)

# ==============================
# TRAIN / VALIDATION SPLIT
# ==============================

train_df,val_df = train_test_split(
    df,
    test_size=0.35,
    random_state=42,
    stratify=df["label"]
)

print("Train Images:",len(train_df))
print("Validation Images:",len(val_df))

# ==============================
# IMAGE LOADING FUNCTION
# ==============================

def load_image(path,label):

    img = tf.io.read_file(path)

    img = tf.image.decode_jpeg(img,channels=3)

    img = tf.image.resize(img,IMG_SIZE)

    img = img/255.0

    return img,label

# ==============================
# TRAIN DATASET
# ==============================

train_dataset = tf.data.Dataset.from_tensor_slices(
    (train_df["image_path"].values,train_df["label"].values)
)

train_dataset = (
    train_dataset
    .map(load_image,num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# ==============================
# VALIDATION DATASET
# ==============================

val_dataset = tf.data.Dataset.from_tensor_slices(
    (val_df["image_path"].values,val_df["label"].values)
)

val_dataset = (
    val_dataset
    .map(load_image,num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# ==============================
# LOAD PRETRAINED RESNET50
# ==============================

base_model = tf.keras.applications.ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(96,96,3)
)

for layer in base_model.layers:
    layer.trainable = False

# ==============================
# CLASSIFICATION HEAD
# ==============================

x = base_model.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dense(256,activation="relu")(x)
x = tf.keras.layers.Dropout(0.4)(x)

outputs = tf.keras.layers.Dense(num_classes,activation="softmax")(x)

model = tf.keras.Model(inputs=base_model.input,outputs=outputs)

# ==============================
# COMPILE MODEL
# ==============================

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# ==============================
# CALLBACKS
# ==============================

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "best_indofashion_model.h5",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

# ==============================
# TRAIN MODEL
# ==============================

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=[early_stop,checkpoint]
)

# ==============================
# SAVE FINAL MODEL
# ==============================

model.save("indofashion_final_model.h5")

print("Training Completed Successfully")

Total Images Before Sampling: 91166
Total Images After 50% Sampling: 45585
Sample Path: images/train/11415.jpeg
Exists: True
Classes: ['blouse' 'dhoti_pants' 'dupattas' 'gowns' 'kurta_men'
 'leggings_and_salwars' 'lehenga' 'mojaris_men' 'mojaris_women'
 'nehru_jackets' 'palazzos' 'petticoats' 'saree' 'sherwanis' 'women_kurta']
Train Images: 29630
Validation Images: 15955


C:\Users\Fixity\AppData\Local\Temp\ipykernel_20004\2249741439.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("class_label").apply(lambda x: x.sample(frac=0.5,random_state=42)).reset_index(drop=True)


Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_2 (InputLayer)        [(None, 96, 96, 3)]          0         []                            
                                                                                                  
 conv1_pad (ZeroPadding2D)   (None, 102, 102, 3)          0         ['input_2[0][0]']             
                                                                                                  
 conv1_conv (Conv2D)         (None, 48, 48, 64)           9472      ['conv1_pad[0][0]']           
                                                                                                  
 conv1_bn (BatchNormalizati  (None, 48, 48, 64)           256       ['conv1_conv[0][0]']          
 on)                                                                                        

c:\Users\Fixity\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


463/463 [==============================] - 705s 2s/step - loss: 1.5121 - accuracy: 0.5291 - val_loss: 1.3709 - val_accuracy: 0.6150
Epoch 2/50
463/463 [==============================] - ETA: 0s - loss: 1.1955 - accuracy: 0.6207
Epoch 2: val_loss improved from 1.37086 to 1.08364, saving model to best_indofashion_model.h5
463/463 [==============================] - 881s 2s/step - loss: 1.1955 - accuracy: 0.6207 - val_loss: 1.0836 - val_accuracy: 0.6628
Epoch 3/50
463/463 [==============================] - ETA: 0s - loss: 1.0951 - accuracy: 0.6513
Epoch 3: val_loss improved from 1.08364 to 1.03738, saving model to best_indofashion_model.h5
463/463 [==============================] - 810s 2s/step - loss: 1.0951 - accuracy: 0.6513 - val_loss: 1.0374 - val_accuracy: 0.6759
Epoch 4/50
463/463 [==============================] - ETA: 0s - loss: 1.0331 - accuracy: 0.6705
Epoch 4: val_loss improved from 1.03738 to 1.01456, saving model to best_indofashion_model.h5
463/463 [=========================

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'])
plt.title("Training Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.show()

In [ ]:
model.save("fashion_classifier.h5")
model.save("fashion_classifier.keras")